# A02a - LLM-based Agents

**Level:** Advanced | **Guest Lecture:** Introduction to LLM-based Agents  
**Speaker:** Ziqin Zhu, University of Auckland (May 2026)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand what an LLM-based agent is and how it differs from a chatbot
2. Understand the ReAct (Reason + Act) pattern and agent loop
3. Know the major reasoning and planning patterns (ReAct, Plan-then-Execute, Reflexion)
4. Understand tool calling mechanics and implement tool definitions
5. Distinguish short-term vs long-term memory in agents
6. Understand skills as progressive disclosure of capabilities
7. Recognize where single agents break down and how to mitigate
8. Build a complete working ReAct agent from scratch

---

## Prerequisites

- Completed A02 (Prompt Engineering and In-Context Learning)
- Familiarity with LLM APIs (OpenAI, etc.)
- Python programming experience

**Duration:** 2-3 hours

---

In [ ]:
# Setup - Install dependencies (uncomment for Colab)
# !pip install openai tiktoken matplotlib -q

import json
import time
import random
import textwrap
from typing import Any, Callable
from dataclasses import dataclass, field

print("Setup complete. Ready to build agents!")

## 1. What is an Agent?

> **An agent is an LLM that runs in a loop: it reasons, acts, observes, and repeats until the task is done.**

### The Agent Equation

```
Agent = LLM + Reasoning + Memory + Planning + Tools + Skills
```

A bare LLM generates text given a prompt. An **agent** wraps that LLM in a loop that gives it:

| Component | What it adds |
|-----------|-------------|
| **Reasoning** | The model thinks step-by-step before acting |
| **Memory** | Short-term (context window) + long-term (files, vector stores) |
| **Planning** | Decompose complex goals into sub-tasks |
| **Tools** | Functions the agent can call (search, calculate, write files) |
| **Skills** | Packaged instructions + tools loaded on demand |

### The ReAct Pattern (Reason + Act)

The most common agent architecture. At each step the model:

```
┌─────────────────────────────────────────┐
│                                         │
│   ┌──────────┐    ┌──────────┐         │
│   │  REASON  │───▶│   ACT    │         │
│   │ (think)  │    │ (tool)   │         │
│   └──────────┘    └────┬─────┘         │
│        ▲                │               │
│        │                ▼               │
│   ┌────┴─────┐    ┌──────────┐         │
│   │  APPEND  │◀───│ OBSERVE  │         │
│   │(to context)   │ (result) │         │
│   └──────────┘    └──────────┘         │
│                                         │
│         Repeat until done               │
└─────────────────────────────────────────┘
```

The key insight: **the LLM drives the loop**, not the user. The agent decides what to do next based on what it has observed so far.

## 2. Agents You Already Use

Before diving into the mechanics, recognize that you likely already interact with agents daily:

### Coding Agents
- **Claude Code** — runs in the terminal, operates across an entire repo
- **Cursor** — agent embedded in the IDE
- **GitHub Copilot** — turns an issue into a pull request autonomously

These agents read the repo, edit files, run tests, and open pull requests.

### General Assistants
- **ChatGPT** and **Claude** — when they call tools and run many steps
- **OpenClaw** — open source, runs locally through your chat apps
- **Hermes** — open-source demo agent (used in the lecture demo)

These chat, then call tools, and run many steps to finish a task.

### Deep Research
- **OpenAI Deep Research** and **Gemini Deep Research** — one agent plans, browses many pages, then writes a sourced report

---

## 3. Chatbot vs Agent

The same LLM model can power both a chatbot and an agent. The difference is the **scaffolding** around it.

| Dimension | Chatbot | Agent |
|-----------|---------|-------|
| **Goal** | Answer questions | Accomplish tasks |
| **Turns** | Single turn (or simple multi-turn) | Multi-step, autonomous |
| **Side effects** | None (just text out) | Calls tools, edits files, sends emails |
| **Who drives?** | User drives the flow | LLM drives the flow |
| **Loop** | No loop — respond and stop | Loop until task complete |
| **Tools** | None (or very limited) | Rich toolset |

### Same Model, Different Wrapper

```python
# Chatbot: one call, one response
response = llm.chat(messages=[{"role": "user", "content": "What is Python?"}])
print(response)  # Done!

# Agent: loop until task is complete
while not done:
    thought = llm.chat(messages=context)      # Reason
    action = parse_tool_call(thought)          # Act
    result = execute_tool(action)              # Observe
    context.append(result)                     # Append
    done = check_if_finished(thought)          # Repeat?
```

The agent adds: **loop + tools + freedom to act**.

## 4. What One Step Looks Like

Each iteration of the agent loop has four phases:

### Step-by-Step Breakdown

```
┌─────────────────────────────────────────────────────────────┐
│ STEP N                                                       │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│ 1. REASON   → Model reads context, picks next move          │
│              "I need to find the weather in Auckland"        │
│                                                              │
│ 2. ACT      → Model emits a tool call with arguments        │
│              tool: get_weather, args: {"city": "Auckland"}   │
│                                                              │
│ 3. OBSERVE  → Your code runs the tool, returns result       │
│              "Auckland: 18°C, partly cloudy"                 │
│                                                              │
│ 4. APPEND   → Result added to context as new message        │
│              [{"role": "tool", "content": "18°C..."}]       │
│                                                              │
│ 5. REPEAT   → Back to step 1 with updated context           │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### Key Points

- The **model never executes code** — it only emits structured requests
- **Your code** is the runtime that actually calls functions
- The model sees tool results as text, just like any other message
- The loop continues until the model decides to respond without a tool call

## 5. Reasoning and Planning Patterns

Three dominant patterns for how agents think and plan:

### 4.1 ReAct (Reason + Act)

> Think → Act → Observe → Repeat

The simplest and most common pattern. The agent interleaves reasoning with action at every step.

```
Thought: I need to find the population of New Zealand
Action: search("population of New Zealand 2024")
Observation: 5.2 million
Thought: Now I have the answer
Action: respond("The population of NZ is approximately 5.2 million")
```

**Pros:** Simple, flexible, works for most tasks  
**Cons:** Can lose track of long-term goals, no explicit plan

---

### 4.2 Plan-then-Execute

> Write full plan first → Execute step by step → Re-plan if reality differs

```
Plan:
1. Search for NZ population
2. Search for NZ GDP
3. Calculate GDP per capita
4. Format and respond

Executing step 1... ✓
Executing step 2... ✓
Executing step 3... ✓ (result differs from expectation, re-planning...)
Revised plan: ...
```

**Pros:** Better for complex multi-step tasks, explicit progress tracking  
**Cons:** Upfront plan may be wrong, overhead of planning step

---

### 4.3 Reflexion

> Try → Critique yourself → Keep notes → Retry with lessons learned

```
Attempt 1: Wrote code, but tests failed
Self-critique: "I forgot to handle the edge case of empty input"
Memory note: "Always check for empty input in string functions"

Attempt 2: Wrote code with empty-input check → tests pass ✓
```

**Pros:** Learns across attempts, self-improving  
**Cons:** Expensive (multiple full attempts), needs clear success signal

---

### Comparison Table

| Pattern | When to Use | Complexity | Best For |
|---------|------------|------------|----------|
| ReAct | Default choice | Low | General tasks, Q&A with tools |
| Plan-then-Execute | Complex multi-step | Medium | Research, data pipelines |
| Reflexion | Trial-and-error tasks | High | Code generation, optimization |

In [ ]:
# Visualizing the three planning patterns

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- ReAct ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("ReAct\n(Think → Act → Observe → Repeat)", fontsize=11, fontweight='bold')
ax.axis('off')

steps = [("Think", 5, 9, '#4ECDC4'), ("Act", 5, 7, '#FF6B6B'), 
         ("Observe", 5, 5, '#45B7D1'), ("Think", 5, 3, '#4ECDC4'), ("Act", 5, 1, '#FF6B6B')]
for label, x, y, color in steps:
    ax.add_patch(mpatches.FancyBboxPatch((x-1.5, y-0.5), 3, 1, 
                 boxstyle="round,pad=0.1", facecolor=color, alpha=0.7))
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
for i in range(len(steps)-1):
    ax.annotate('', xy=(5, steps[i+1][2]+0.5), xytext=(5, steps[i][2]-0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# --- Plan-then-Execute ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("Plan-then-Execute\n(Plan → Step 1 → Step 2 → ...)", fontsize=11, fontweight='bold')
ax.axis('off')

steps2 = [("PLAN", 5, 9, '#FFE66D'), ("Step 1", 5, 7, '#95E1D3'), 
          ("Step 2", 5, 5, '#95E1D3'), ("Step 3", 5, 3, '#95E1D3'), ("Re-plan?", 5, 1, '#FFE66D')]
for label, x, y, color in steps2:
    ax.add_patch(mpatches.FancyBboxPatch((x-1.5, y-0.5), 3, 1,
                 boxstyle="round,pad=0.1", facecolor=color, alpha=0.7))
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
for i in range(len(steps2)-1):
    ax.annotate('', xy=(5, steps2[i+1][2]+0.5), xytext=(5, steps2[i][2]-0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
# Re-plan arrow back to top
ax.annotate('', xy=(7.5, 9), xytext=(7.5, 1),
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.5, linestyle='dashed'))

# --- Reflexion ---
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("Reflexion\n(Try → Critique → Learn → Retry)", fontsize=11, fontweight='bold')
ax.axis('off')

steps3 = [("Attempt", 5, 9, '#FF6B6B'), ("Evaluate", 5, 7, '#45B7D1'),
           ("Self-Critique", 5, 5, '#FFE66D'), ("Memory Note", 5, 3, '#C9B1FF'), ("Retry", 5, 1, '#4ECDC4')]
for label, x, y, color in steps3:
    ax.add_patch(mpatches.FancyBboxPatch((x-1.8, y-0.5), 3.6, 1,
                 boxstyle="round,pad=0.1", facecolor=color, alpha=0.7))
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')
for i in range(len(steps3)-1):
    ax.annotate('', xy=(5, steps3[i+1][2]+0.5), xytext=(5, steps3[i][2]-0.5),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(8, 9), xytext=(8, 1),
            arrowprops=dict(arrowstyle='->', color='green', lw=1.5, linestyle='dashed'))
ax.text(8.5, 5, 'improved', fontsize=8, color='green', rotation=90, va='center')

plt.tight_layout()
plt.savefig('agent_patterns.png', dpi=100, bbox_inches='tight')
plt.show()
print("Three major reasoning/planning patterns for LLM agents.")

## 6. Tool Calling

Tools give agents the ability to **do things** beyond generating text. The mechanics:

### The Four Steps of Tool Calling

```
┌──────────────────────────────────────────────────────────────┐
│                                                              │
│  1. DEFINE    → You describe tools (name, description,      │
│                 JSON input schema) in the system prompt      │
│                                                              │
│  2. DECIDE    → Model reads the task and emits a            │
│                 structured tool call (not free text)         │
│                                                              │
│  3. RUN       → YOUR CODE executes the function             │
│                 (model never runs it directly!)              │
│                                                              │
│  4. RETURN    → Pass the result back as a new message       │
│                 for the model to reason about                │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

### Example Tool Definition (OpenAI Format)

```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get current weather for a city",
    "parameters": {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "City name, e.g. Auckland"
        },
        "units": {
          "type": "string",
          "enum": ["celsius", "fahrenheit"],
          "description": "Temperature units"
        }
      },
      "required": ["city"]
    }
  }
}
```

### Key Insight

> The model **never executes** the tool. It only outputs a structured request.  
> Your application code is the runtime that actually calls the function and returns the result.

In [ ]:
# Tool Calling Implementation

from dataclasses import dataclass
from typing import Any, Callable
import json

@dataclass
class ToolDefinition:
    """Defines a tool that an agent can call."""
    name: str
    description: str
    parameters: dict  # JSON Schema
    function: Callable  # The actual implementation

@dataclass 
class ToolCall:
    """A structured tool call emitted by the model."""
    name: str
    arguments: dict

@dataclass
class ToolResult:
    """Result returned after executing a tool."""
    tool_name: str
    result: str
    success: bool = True


# --- Define some example tools ---

def get_weather(city: str, units: str = "celsius") -> str:
    """Simulated weather lookup."""
    weather_data = {
        "auckland": {"temp": 18, "condition": "partly cloudy"},
        "london": {"temp": 12, "condition": "rainy"},
        "tokyo": {"temp": 24, "condition": "sunny"},
        "new york": {"temp": 20, "condition": "clear"},
    }
    data = weather_data.get(city.lower(), {"temp": 15, "condition": "unknown"})
    unit_symbol = "°C" if units == "celsius" else "°F"
    temp = data["temp"] if units == "celsius" else int(data["temp"] * 9/5 + 32)
    return f"{city}: {temp}{unit_symbol}, {data['condition']}"

def calculate(expression: str) -> str:
    """Safe calculator - evaluates math expressions."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "Error: only basic math operations allowed"
    try:
        result = eval(expression)  # Safe because we validated input
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def search_knowledge(query: str) -> str:
    """Simulated knowledge base search."""
    knowledge = {
        "population new zealand": "New Zealand population (2024): ~5.2 million",
        "capital france": "The capital of France is Paris",
        "python creator": "Python was created by Guido van Rossum in 1991",
        "largest ocean": "The Pacific Ocean is the largest ocean on Earth",
    }
    for key, value in knowledge.items():
        if any(word in query.lower() for word in key.split()):
            return value
    return f"No results found for: {query}"


# Register tools with their schemas
TOOLS = [
    ToolDefinition(
        name="get_weather",
        description="Get current weather for a city",
        parameters={
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name"},
                "units": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}
            },
            "required": ["city"]
        },
        function=get_weather
    ),
    ToolDefinition(
        name="calculate",
        description="Evaluate a mathematical expression",
        parameters={
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Math expression, e.g. '2 + 2'"}
            },
            "required": ["expression"]
        },
        function=calculate
    ),
    ToolDefinition(
        name="search_knowledge",
        description="Search a knowledge base for factual information",
        parameters={
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        },
        function=search_knowledge
    ),
]

# Tool registry for quick lookup
tool_registry = {tool.name: tool for tool in TOOLS}

# Demonstrate tool execution
print("=== Tool Calling Demo ===\n")

# Simulate what happens when model emits a tool call
tool_call = ToolCall(name="get_weather", arguments={"city": "Auckland", "units": "celsius"})
print(f"Model emits: tool={tool_call.name}, args={tool_call.arguments}")

# Your code executes it
tool = tool_registry[tool_call.name]
result = tool.function(**tool_call.arguments)
print(f"Tool returns: {result}")
print(f"\nThis result goes back into the context as a new message for the model.")

print("\n--- All registered tools ---")
for t in TOOLS:
    print(f"  • {t.name}: {t.description}")

## 7. Memory: Short-term and Long-term

Agents need memory to maintain context and learn from past interactions.

### Two Types of Memory

```
┌─────────────────────────────────────────────────────────────────┐
│                        AGENT MEMORY                              │
├────────────────────────────┬────────────────────────────────────┤
│     SHORT-TERM MEMORY      │       LONG-TERM MEMORY             │
├────────────────────────────┼────────────────────────────────────┤
│ • Running transcript       │ • Files on disk                    │
│ • In the context window    │ • Database records                 │
│ • Volatile                 │ • Vector store embeddings          │
│ • Gone when run ends       │ • Recalled on demand               │
│ • Limited by token count   │ • Survives across runs             │
│ • Always "loaded"          │ • Must be explicitly retrieved     │
├────────────────────────────┼────────────────────────────────────┤
│ Like human working memory  │ Like notes in a notebook           │
└────────────────────────────┴────────────────────────────────────┘
```

### Short-term Memory (Context Window)

- Every message in the conversation: user messages, assistant responses, tool results
- The model sees ALL of this on every turn
- Limited by context window size (4K, 8K, 128K, 1M tokens)
- When it fills up: summarize, truncate, or use sliding window

### Long-term Memory (Persistent Storage)

- **Files:** Save notes, plans, or results to disk
- **Database:** Store structured data (user preferences, task history)
- **Vector Store:** Embed and retrieve relevant past experiences
- **Key-Value Store:** Quick lookup of facts learned previously

### When to Use Each

| Situation | Memory Type | Example |
|-----------|-------------|---------|
| Current conversation | Short-term | "User asked about weather" |
| User preferences | Long-term (DB) | "User prefers Celsius" |
| Past research | Long-term (Vector) | "Previously found NZ pop = 5.2M" |
| Task progress | Short-term + Long-term | Save checkpoints to file |

In [ ]:
# Memory Management Implementation

from dataclasses import dataclass, field
from typing import List, Optional
import json
import time

@dataclass
class Message:
    """A single message in the conversation."""
    role: str  # "system", "user", "assistant", "tool"
    content: str
    tool_call: Optional[dict] = None
    timestamp: float = field(default_factory=time.time)

class ShortTermMemory:
    """Context window management - the running transcript."""
    
    def __init__(self, max_messages: int = 50):
        self.messages: List[Message] = []
        self.max_messages = max_messages
    
    def add(self, message: Message):
        self.messages.append(message)
        # Simple truncation strategy: keep system + recent messages
        if len(self.messages) > self.max_messages:
            system_msgs = [m for m in self.messages if m.role == "system"]
            recent = self.messages[-(self.max_messages - len(system_msgs)):]
            self.messages = system_msgs + recent
            print(f"  [Memory] Truncated to {len(self.messages)} messages")
    
    def get_context(self) -> List[dict]:
        """Return messages in API format."""
        return [{"role": m.role, "content": m.content} for m in self.messages]
    
    def clear(self):
        self.messages = []
    
    @property
    def token_estimate(self) -> int:
        """Rough token count (4 chars ≈ 1 token)."""
        return sum(len(m.content) for m in self.messages) // 4


class LongTermMemory:
    """Persistent memory that survives across runs."""
    
    def __init__(self):
        self.store: dict = {}  # Key-value store
        self.notes: List[str] = []  # Free-form notes
    
    def save_fact(self, key: str, value: str):
        """Store a fact for later retrieval."""
        self.store[key] = {"value": value, "timestamp": time.time()}
        print(f"  [LTM] Saved: {key} = {value}")
    
    def recall_fact(self, key: str) -> Optional[str]:
        """Retrieve a previously stored fact."""
        if key in self.store:
            return self.store[key]["value"]
        return None
    
    def add_note(self, note: str):
        """Add a free-form note (like Reflexion memory)."""
        self.notes.append(note)
        print(f"  [LTM] Note added: {note}")
    
    def search_notes(self, query: str) -> List[str]:
        """Simple keyword search over notes."""
        return [n for n in self.notes if query.lower() in n.lower()]
    
    def export(self) -> str:
        """Export memory to JSON (for persistence to disk)."""
        return json.dumps({"store": self.store, "notes": self.notes}, indent=2)


# Demo
print("=== Memory Demo ===\n")

stm = ShortTermMemory(max_messages=5)
ltm = LongTermMemory()

# Short-term: conversation flows in
stm.add(Message(role="system", content="You are a helpful assistant."))
stm.add(Message(role="user", content="What's the weather in Auckland?"))
stm.add(Message(role="assistant", content="Let me check that for you."))
print(f"Short-term memory: {len(stm.messages)} messages, ~{stm.token_estimate} tokens\n")

# Long-term: persist facts across runs
ltm.save_fact("user_preference_units", "celsius")
ltm.save_fact("nz_population", "5.2 million")
ltm.add_note("User prefers concise answers")
ltm.add_note("Previous task: weather lookup succeeded")

print(f"\nRecall 'user_preference_units': {ltm.recall_fact('user_preference_units')}")
print(f"Search notes for 'weather': {ltm.search_notes('weather')}")

## 8. Skills

> A **skill** is a folder with instructions plus optional scripts/tools.

### The Progressive Disclosure Pattern

Loading every possible instruction into the context window wastes tokens and confuses the model. Skills solve this with **progressive disclosure**:

```
┌─────────────────────────────────────────────────────────────┐
│ AT STARTUP: Only name + description loaded                   │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  skills/                                                     │
│  ├── web-search/                                            │
│  │   ├── instructions.md    ← Full instructions             │
│  │   ├── search_tool.py     ← Custom tool code              │
│  │   └── manifest.json      ← Name + description only       │
│  ├── code-review/                                           │
│  │   ├── instructions.md                                    │
│  │   ├── lint_rules.yaml                                    │
│  │   └── manifest.json                                      │
│  └── data-analysis/                                         │
│      ├── instructions.md                                    │
│      ├── plot_helpers.py                                    │
│      └── manifest.json                                      │
│                                                              │
├─────────────────────────────────────────────────────────────┤
│ WHEN TASK MATCHES: Full instructions loaded into context     │
└─────────────────────────────────────────────────────────────┘
```

### How It Works

1. **At startup:** Agent sees a list of skill names and one-line descriptions
2. **When a task arrives:** Agent matches the task to a skill
3. **On match:** Full instructions + tools are loaded into context
4. **Benefit:** Context stays lean until expertise is needed

### Example Manifest

```json
{
  "name": "web-search",
  "description": "Search the web and summarize results for factual queries",
  "triggers": ["search", "find", "look up", "what is"],
  "tools": ["web_search", "summarize_page"]
}
```

This is exactly how modern agent frameworks (Kiro, Cursor, etc.) manage capabilities at scale.

In [ ]:
# Skills Implementation - Progressive Disclosure

from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Skill:
    """A skill that can be loaded on demand."""
    name: str
    description: str  # Short - always in context
    triggers: List[str]  # Keywords that activate this skill
    full_instructions: str  # Only loaded when activated
    tools: List[str] = field(default_factory=list)
    _loaded: bool = False

    def matches(self, task: str) -> bool:
        """Check if this skill matches the given task."""
        task_lower = task.lower()
        return any(trigger in task_lower for trigger in self.triggers)
    
    def activate(self) -> str:
        """Load full instructions into context."""
        self._loaded = True
        return self.full_instructions


class SkillRegistry:
    """Manages available skills with progressive disclosure."""
    
    def __init__(self):
        self.skills: List[Skill] = []
    
    def register(self, skill: Skill):
        self.skills.append(skill)
    
    def get_manifest(self) -> str:
        """Return compact manifest for system prompt (always loaded)."""
        lines = ["Available skills:"]
        for s in self.skills:
            lines.append(f"  • {s.name}: {s.description}")
        return "\n".join(lines)
    
    def match_and_load(self, task: str) -> Optional[str]:
        """Find matching skill and return its full instructions."""
        for skill in self.skills:
            if skill.matches(task):
                print(f"  [Skills] Activated: {skill.name}")
                return skill.activate()
        return None


# Create some example skills
from dataclasses import field

registry = SkillRegistry()

registry.register(Skill(
    name="web-search",
    description="Search the web for current information",
    triggers=["search", "find", "look up", "current", "latest"],
    full_instructions="""## Web Search Skill

When searching the web:
1. Formulate a clear, specific search query
2. Use the search_web tool with the query
3. Read the top 3 results
4. Synthesize information from multiple sources
5. Cite your sources

Always prefer recent sources (< 1 year old).
If results conflict, note the disagreement.""",
    tools=["search_web", "read_page"]
))

registry.register(Skill(
    name="code-review",
    description="Review code for bugs, style, and best practices",
    triggers=["review", "check code", "bugs", "code quality"],
    full_instructions="""## Code Review Skill

When reviewing code:
1. Check for common bugs (off-by-one, null refs, race conditions)
2. Evaluate naming and readability
3. Check error handling
4. Look for security issues (injection, auth)
5. Suggest improvements with examples

Format: list issues by severity (critical > warning > suggestion).""",
    tools=["read_file", "lint_check"]
))

registry.register(Skill(
    name="data-analysis",
    description="Analyze datasets and create visualizations",
    triggers=["analyze", "plot", "chart", "statistics", "data"],
    full_instructions="""## Data Analysis Skill

When analyzing data:
1. Load and inspect the dataset (shape, types, missing values)
2. Compute summary statistics
3. Create appropriate visualizations
4. Identify patterns and outliers
5. Summarize findings in plain language

Always show your work with code.""",
    tools=["load_csv", "compute_stats", "create_plot"]
))

# Demo: what the agent sees at startup (compact)
print("=== What the agent sees at startup ===")
print(registry.get_manifest())

# Demo: task comes in, skill is matched and loaded
print("\n=== Task arrives: 'Please review my code for bugs' ===")
instructions = registry.match_and_load("Please review my code for bugs")
print(f"\nLoaded instructions:\n{instructions[:200]}...")

## 9. Where a Single Agent Breaks

Single agents have real limitations. Understanding these helps you design better systems.

### 8.1 Lost-in-the-Middle

> **20%+ accuracy drop** when relevant information is in the middle of a long context.

```
Context: [start ... ... RELEVANT INFO ... ... end]
                         ↑
              Model often misses this!
```

LLMs attend strongly to the beginning and end of context, but struggle with information buried in the middle (Liu et al., TACL 2024).

### 8.2 Plan Drift

> Errors compound. The agent self-conditions on its own mistakes.

```
Step 1: Correct ✓
Step 2: Small error ✗ (based on step 1)
Step 3: Bigger error ✗✗ (based on wrong step 2)
Step 4: Completely off track ✗✗✗
```

Once the agent makes a mistake, subsequent reasoning builds on that mistake, making recovery harder with each step.

### 8.3 Tool Confusion

> Large API pools → hallucinated tool calls

When an agent has access to dozens of tools, it may:
- Call tools that don't exist (hallucinated names)
- Pass wrong argument types
- Use the wrong tool for the job
- Chain tools in nonsensical ways

### Solutions

| Problem | Solution |
|---------|----------|
| Lost-in-the-middle | **Divide and conquer** — split context into chunks, process separately |
| Plan drift | **Replanning** — periodically re-evaluate and correct course |
| Tool confusion | **Smaller, precise toolset** — fewer tools, better descriptions |

### Additional Mitigations

```
┌─────────────────────────────────────────────────────────────┐
│ MULTI-AGENT ARCHITECTURE                                     │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  Orchestrator Agent                                          │
│       │                                                      │
│       ├── Research Agent (small toolset: search, read)       │
│       ├── Code Agent (small toolset: edit, run, test)        │
│       └── Review Agent (small toolset: read, comment)        │
│                                                              │
│  Each sub-agent has a focused context and few tools          │
│  → Less confusion, less drift, shorter contexts             │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# Visualizing the "Lost in the Middle" phenomenon

import matplotlib.pyplot as plt
import numpy as np

# Simulated accuracy based on position of relevant info in context
# Based on findings from Liu et al., 2023
positions = np.arange(1, 21)  # 20 positions in context
# U-shaped curve: high at start and end, low in middle
accuracy = 75 + 20 * np.exp(-0.3 * (positions - 1)) + 15 * np.exp(-0.3 * (20 - positions))
accuracy = np.clip(accuracy, 55, 98)
# Add some noise for realism
np.random.seed(42)
accuracy += np.random.normal(0, 1.5, len(positions))
accuracy = np.clip(accuracy, 50, 100)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(positions, accuracy, 'o-', color='#FF6B6B', linewidth=2, markersize=8)
ax.fill_between(positions, accuracy, alpha=0.1, color='#FF6B6B')
ax.axhline(y=75, color='gray', linestyle='--', alpha=0.5, label='Baseline (no distraction)')
ax.axvspan(7, 14, alpha=0.1, color='red', label='Danger zone (middle)')

ax.set_xlabel('Position of Relevant Information in Context', fontsize=12)
ax.set_ylabel('Model Accuracy (%)', fontsize=12)
ax.set_title('Lost-in-the-Middle: Accuracy Drops for Mid-Context Information', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(45, 105)
ax.set_xticks(positions)
ax.grid(True, alpha=0.3)

# Annotate
ax.annotate('~20% drop!', xy=(10, accuracy[9]), xytext=(12, 58),
            fontsize=11, fontweight='bold', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()
print("Key insight: Place critical information at the START or END of context, not the middle.")

## 10. Building Your Own: Two Paths

### Code-First Frameworks

For maximum control and customization:

| Framework | By | Best For |
|-----------|-----|----------|
| **OpenAI Agents SDK** | OpenAI | Quick start, native tool calling |
| **LangGraph** | LangChain | Complex stateful workflows, graphs |
| **Microsoft Agent Framework** | Microsoft | Successor to Semantic Kernel and AutoGen; .NET and Python |
| **CrewAI** | Community | Multi-agent collaboration |

```python
# Example: OpenAI Agents SDK (simplified)
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tool_definitions,  # Your tool schemas
    tool_choice="auto"       # Let model decide
)
```

### Low-Code / No-Code Platforms

For rapid prototyping and non-developers:

| Platform | Strength |
|----------|----------|
| **Dify** | Visual workflow builder, RAG built-in |
| **n8n** | Automation workflows, 400+ integrations |
| **Flowise** | Drag-and-drop LangChain flows |
| **Langflow** | Visual LangChain builder |

### Which Path to Choose?

```
Need full control over agent behavior?     → Code-first
Need to ship fast / prototype?             → Low-code
Need custom reasoning patterns?            → Code-first
Need non-technical team to build?          → Low-code
Need to handle edge cases precisely?       → Code-first
Need integrations with existing tools?     → Low-code (n8n)
```

### Getting Started Links

**Code-first:**
- OpenAI Agents SDK: `openai.github.io/openai-agents-python`
- LangGraph: `langchain-ai.github.io/langgraph`
- Microsoft Agent Framework: `learn.microsoft.com/agent-framework`

**Low-code:**
- Dify: `dify.ai`
- n8n: `n8n.io`
- Flowise: `flowiseai.com`

## 11. Practical Implementation: Building a ReAct Agent

Now let's build a **complete, working ReAct agent** from scratch. This agent will:

1. Accept a user task
2. Reason about what to do (using a mock LLM or real API)
3. Call tools when needed
4. Observe results
5. Loop until the task is complete

We'll build it in layers:
1. **Mock LLM** — deterministic "model" for testing without API keys
2. **Agent Loop** — the core ReAct cycle
3. **Full Demo** — running the agent on real tasks

In [ ]:
# ============================================================
# COMPLETE ReAct AGENT IMPLEMENTATION
# ============================================================

import json
import re
from dataclasses import dataclass, field
from typing import List, Optional, Callable, Dict, Any
import time

# --- Data Classes ---

@dataclass
class AgentMessage:
    role: str  # "system", "user", "assistant", "tool"
    content: str
    tool_call: Optional[Dict] = None
    tool_result: Optional[str] = None

@dataclass
class AgentTool:
    name: str
    description: str
    parameters: Dict
    function: Callable

# --- Mock LLM (deterministic, no API key needed) ---

class MockLLM:
    """
    A rule-based mock LLM that simulates ReAct reasoning.
    Replace with a real API call for production use.
    """
    
    def __init__(self, tools: List[AgentTool]):
        self.tools = {t.name: t for t in tools}
        self.tool_list = tools
    
    def generate(self, messages: List[AgentMessage]) -> AgentMessage:
        """Simulate LLM response based on conversation state."""
        
        # Get the last user message and any tool results
        last_user_msg = ""
        last_tool_result = None
        pending_tool_calls = 0
        
        for msg in messages:
            if msg.role == "user":
                last_user_msg = msg.content
            if msg.role == "tool":
                last_tool_result = msg.content
                pending_tool_calls = 0
            if msg.role == "assistant" and msg.tool_call:
                pending_tool_calls += 1
        
        # If we just got a tool result, synthesize an answer
        if last_tool_result and pending_tool_calls == 0:
            return AgentMessage(
                role="assistant",
                content=f"Based on my research: {last_tool_result}"
            )
        
        # Determine what tool to call based on the user's question
        user_lower = last_user_msg.lower()
        
        if "weather" in user_lower:
            # Extract city
            city = "Auckland"  # default
            for c in ["auckland", "london", "tokyo", "new york", "paris"]:
                if c in user_lower:
                    city = c.title()
                    break
            return AgentMessage(
                role="assistant",
                content=f"Thought: I need to find the weather in {city}. I'll use the get_weather tool.",
                tool_call={"name": "get_weather", "arguments": {"city": city, "units": "celsius"}}
            )
        
        elif any(word in user_lower for word in ["calculate", "math", "compute", "what is", "how much"]):
            # Try to extract a math expression
            # Look for patterns like "what is 25 * 4" or "calculate 100 / 5"
            math_match = re.search(r'[\d][\d\s+\-*/().]+[\d]', last_user_msg)
            if math_match:
                expr = math_match.group().strip()
                return AgentMessage(
                    role="assistant",
                    content=f"Thought: I need to calculate {expr}. I'll use the calculator.",
                    tool_call={"name": "calculate", "arguments": {"expression": expr}}
                )
        
        elif any(word in user_lower for word in ["population", "capital", "who", "creator", "largest"]):
            return AgentMessage(
                role="assistant",
                content=f"Thought: I need to search for this information.",
                tool_call={"name": "search_knowledge", "arguments": {"query": last_user_msg}}
            )
        
        # Default: just respond
        return AgentMessage(
            role="assistant",
            content=f"I can help you with weather lookups, calculations, and knowledge searches. What would you like to know?"
        )


# --- The ReAct Agent ---

class ReActAgent:
    """
    A complete ReAct agent that reasons, acts, and observes in a loop.
    """
    
    def __init__(self, llm, tools: List[AgentTool], max_steps: int = 5, verbose: bool = True):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.max_steps = max_steps
        self.verbose = verbose
        self.messages: List[AgentMessage] = []
        
        # System prompt
        tool_descriptions = "\n".join(
            f"  - {t.name}: {t.description}" for t in tools
        )
        self.messages.append(AgentMessage(
            role="system",
            content=f"""You are a helpful assistant that uses tools to answer questions.

Available tools:
{tool_descriptions}

Use the ReAct pattern:
1. Thought: reason about what to do
2. Action: call a tool if needed
3. Observation: see the result
4. Repeat or respond with final answer."""
        ))
    
    def run(self, user_input: str) -> str:
        """Run the agent loop on a user input. Returns final response."""
        
        self.messages.append(AgentMessage(role="user", content=user_input))
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"  USER: {user_input}")
            print(f"{'='*60}")
        
        for step in range(1, self.max_steps + 1):
            if self.verbose:
                print(f"\n--- Step {step} ---")
            
            # REASON: Ask the LLM what to do next
            response = self.llm.generate(self.messages)
            self.messages.append(response)
            
            if self.verbose:
                print(f"  Assistant: {response.content}")
            
            # Check if the model wants to call a tool
            if response.tool_call:
                tool_name = response.tool_call["name"]
                tool_args = response.tool_call["arguments"]
                
                if self.verbose:
                    print(f"  Action: {tool_name}({json.dumps(tool_args)})")
                
                # ACT: Execute the tool
                if tool_name in self.tools:
                    tool = self.tools[tool_name]
                    try:
                        result = tool.function(**tool_args)
                    except Exception as e:
                        result = f"Error: {e}"
                else:
                    result = f"Error: Unknown tool '{tool_name}'"
                
                if self.verbose:
                    print(f"  Observation: {result}")
                
                # OBSERVE: Add result to context
                self.messages.append(AgentMessage(
                    role="tool",
                    content=result,
                    tool_result=result
                ))
            else:
                # No tool call = final answer
                if self.verbose:
                    print(f"\n{'='*60}")
                    print(f"  FINAL ANSWER: {response.content}")
                    print(f"{'='*60}")
                return response.content
        
        return "Max steps reached without a final answer."


# --- Create the agent ---

# Define tools (reusing our earlier implementations)
agent_tools = [
    AgentTool(
        name="get_weather",
        description="Get current weather for a city",
        parameters={"city": "string", "units": "string"},
        function=get_weather
    ),
    AgentTool(
        name="calculate",
        description="Evaluate a mathematical expression",
        parameters={"expression": "string"},
        function=calculate
    ),
    AgentTool(
        name="search_knowledge",
        description="Search knowledge base for factual information",
        parameters={"query": "string"},
        function=search_knowledge
    ),
]

# Create LLM and Agent
mock_llm = MockLLM(agent_tools)
agent = ReActAgent(llm=mock_llm, tools=agent_tools, max_steps=5, verbose=True)

print("ReAct Agent created successfully!")
print(f"Tools: {list(agent.tools.keys())}")
print(f"Max steps: {agent.max_steps}")

In [ ]:
# === Demo 1: Weather Query ===
print("\n" + "="*60)
print("DEMO 1: Weather Query")
print("="*60)

agent1 = ReActAgent(llm=MockLLM(agent_tools), tools=agent_tools, max_steps=5, verbose=True)
result = agent1.run("What's the weather like in Tokyo?")

In [ ]:
# === Demo 2: Knowledge Query ===
print("\n" + "="*60)
print("DEMO 2: Knowledge Query")
print("="*60)

agent2 = ReActAgent(llm=MockLLM(agent_tools), tools=agent_tools, max_steps=5, verbose=True)
result = agent2.run("Who created Python?")

In [ ]:
# === Demo 3: Calculation ===
print("\n" + "="*60)
print("DEMO 3: Calculation")
print("="*60)

agent3 = ReActAgent(llm=MockLLM(agent_tools), tools=agent_tools, max_steps=5, verbose=True)
result = agent3.run("Calculate 25 * 4 + 100")

## 11.1 Using a Real LLM (Optional - Requires API Key)

The following cell shows how to swap the MockLLM for a real OpenAI API call. 
This requires an `OPENAI_API_KEY` environment variable.

> **Note:** This cell is optional. The mock agent above demonstrates the same concepts without needing an API key.

In [ ]:
# OPTIONAL: Real OpenAI API Integration
# Uncomment and run if you have an API key set

"""
import os
from openai import OpenAI

class RealLLM:
    def __init__(self, tools: List[AgentTool], model: str = "gpt-4o-mini"):
        self.client = OpenAI()  # Uses OPENAI_API_KEY env var
        self.model = model
        self.tool_schemas = [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": {
                        "type": "object",
                        "properties": {
                            k: {"type": v, "description": k}
                            for k, v in t.parameters.items()
                        },
                        "required": list(t.parameters.keys())
                    }
                }
            }
            for t in tools
        ]
    
    def generate(self, messages: List[AgentMessage]) -> AgentMessage:
        api_messages = [{"role": m.role, "content": m.content} for m in messages]
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=api_messages,
            tools=self.tool_schemas,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        if msg.tool_calls:
            tc = msg.tool_calls[0]
            return AgentMessage(
                role="assistant",
                content=msg.content or f"Calling {tc.function.name}...",
                tool_call={
                    "name": tc.function.name,
                    "arguments": json.loads(tc.function.arguments)
                }
            )
        
        return AgentMessage(role="assistant", content=msg.content)

# Usage:
# real_llm = RealLLM(agent_tools)
# agent = ReActAgent(llm=real_llm, tools=agent_tools, max_steps=5)
# agent.run("What's the weather in Auckland?")
"""

print("Real LLM integration code shown above (commented out).")
print("To use: set OPENAI_API_KEY and uncomment the code.")

In [ ]:
# Complete Agent Architecture Visualization

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Complete LLM Agent Architecture', fontsize=14, fontweight='bold', pad=20)

# Main components
components = [
    # (label, x, y, width, height, color)
    ("LLM\n(Brain)", 7, 7.5, 2.5, 1.5, '#FF6B6B'),
    ("Tools", 2, 7.5, 2, 1.2, '#4ECDC4'),
    ("Short-term\nMemory", 12, 7.5, 2, 1.2, '#45B7D1'),
    ("Long-term\nMemory", 12, 5, 2, 1.2, '#C9B1FF'),
    ("Skills", 2, 5, 2, 1.2, '#FFE66D'),
    ("Planning", 7, 5, 2.5, 1.2, '#95E1D3'),
    ("Agent Loop\n(Orchestrator)", 7, 2.5, 3, 1.5, '#FFB347'),
    ("User", 7, 0.5, 2, 1, '#E8E8E8'),
]

for label, x, y, w, h, color in components:
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='gray',
                                    linewidth=1.5, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows (connections)
arrow_style = dict(arrowstyle='->', color='#555', lw=1.5, connectionstyle='arc3,rad=0.1')

# Agent Loop <-> LLM
ax.annotate('', xy=(7, 6.75), xytext=(7, 3.25), arrowprops=arrow_style)
ax.annotate('', xy=(7, 3.25), xytext=(7, 6.75), arrowprops=dict(arrowstyle='->', color='#555', lw=1.5, connectionstyle='arc3,rad=-0.1'))

# LLM <-> Tools
ax.annotate('', xy=(3, 7.5), xytext=(5.75, 7.5), arrowprops=arrow_style)

# LLM <-> Short-term Memory
ax.annotate('', xy=(11, 7.5), xytext=(8.25, 7.5), arrowprops=arrow_style)

# LLM <-> Planning
ax.annotate('', xy=(7, 6.75), xytext=(7, 5.6), arrowprops=dict(arrowstyle='->', color='#95E1D3', lw=1.5))

# Short-term <-> Long-term Memory
ax.annotate('', xy=(12, 6.9), xytext=(12, 5.6), arrowprops=dict(arrowstyle='<->', color='#888', lw=1.2))

# Skills -> LLM
ax.annotate('', xy=(5.75, 6.9), xytext=(3, 5.6), arrowprops=dict(arrowstyle='->', color='#FFE66D', lw=1.5, connectionstyle='arc3,rad=0.2'))

# User <-> Agent Loop
ax.annotate('', xy=(7, 1.75), xytext=(7, 1.0), arrowprops=dict(arrowstyle='<->', color='#888', lw=2))

# Labels on arrows
ax.text(4.3, 8.0, 'call', fontsize=8, color='#555', style='italic')
ax.text(9.5, 8.0, 'read/write', fontsize=8, color='#555', style='italic')
ax.text(5.5, 2.8, 'reason\n& act', fontsize=8, color='#555', style='italic', ha='center')

plt.tight_layout()
plt.show()
print("The agent loop orchestrates everything: it asks the LLM to reason,")
print("calls tools, manages memory, and loads skills as needed.")

## Summary and Key Takeaways

### The Big Picture

```
┌─────────────────────────────────────────────────────────────┐
│                                                              │
│  An AGENT is just an LLM in a loop with tools.              │
│                                                              │
│  The same model that powers a chatbot can power an agent.   │
│  The difference is the scaffolding: loop + tools + freedom. │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts Recap

| # | Concept | One-liner |
|---|---------|-----------|
| 1 | Agent | LLM + loop + tools + memory |
| 2 | ReAct | Think → Act → Observe → Repeat |
| 3 | Chatbot vs Agent | Same model, different wrapper |
| 4 | Tool Calling | Model emits request, your code executes |
| 5 | Short-term Memory | Context window (volatile) |
| 6 | Long-term Memory | Files/DB/vectors (persistent) |
| 7 | Skills | Progressive disclosure of capabilities |
| 8 | Limitations | Lost-in-middle, plan drift, tool confusion |
| 9 | Solutions | Divide & conquer, replan, focused toolsets |

### What to Build Next

1. **Starter:** Add more tools to the ReAct agent (file I/O, web search)
2. **Intermediate:** Implement Plan-then-Execute with replanning
3. **Advanced:** Build a multi-agent system with an orchestrator
4. **Expert:** Add Reflexion with a test-driven code generation agent

### Recommended Reading

- Yao et al. (2023) — *ReAct: Synergizing Reasoning and Acting in Language Models*, ICLR 2023. [arXiv:2210.03629](https://arxiv.org/abs/2210.03629)
- Shinn et al. (2023) — *Reflexion: Language Agents with Verbal Reinforcement Learning*, NeurIPS 2023. [arXiv:2303.11366](https://arxiv.org/abs/2303.11366)
- Wang et al. (2023) — *Plan-and-Solve Prompting*, ACL 2023. [aclanthology.org/2023.acl-long.147](https://aclanthology.org/2023.acl-long.147)
- Liu et al. (2024) — *Lost in the Middle: How Language Models Use Long Contexts*, TACL 2024
- Sinha et al. (2025) — *Plan Drift in LLM Agents*, [arXiv:2509.09677](https://arxiv.org/abs/2509.09677)
- Qin et al. (2024) — *ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs*, ICLR 2024
- Packer et al. (2023) — *MemGPT: Towards LLMs as Operating Systems*, [arXiv:2310.08560](https://arxiv.org/abs/2310.08560)
- Anthropic (2025) — *Agent Skills*; open standard at [agentskills.io](https://agentskills.io)

---

*Notebook created for COMPSCI 714 — Guest Lecture by Ziqin Zhu, University of Auckland, May 2026*

## Exercises

### Exercise 1: Add a New Tool
Add a `get_time` tool that returns the current time for a given timezone. Register it with the agent and test it.

### Exercise 2: Implement Conversation Memory
Modify the agent so it remembers facts from previous queries in the same session. If the user asks "What about London?" after asking about Auckland's weather, the agent should understand the context.

### Exercise 3: Plan-then-Execute
Implement a `PlanThenExecuteAgent` that:
1. Takes a complex task (e.g., "Compare the weather in 3 cities")
2. Generates a plan (list of steps)
3. Executes each step
4. Combines results into a final answer

### Exercise 4: Reflexion
Implement a simple Reflexion loop:
1. Agent attempts to write a Python function
2. Run the function against test cases
3. If tests fail, agent critiques its own code
4. Agent retries with the critique as additional context
5. Repeat until tests pass or max attempts reached

### Exercise 5: Multi-Agent
Build a simple orchestrator that delegates to specialized sub-agents:
- A "research agent" with search tools
- A "math agent" with calculation tools
- The orchestrator decides which agent to call based on the query

In [ ]:
# Exercise 1 Starter: Add a new tool

from datetime import datetime

def get_time(timezone: str = "UTC") -> str:
    """Get current time. In a real implementation, use pytz or zoneinfo."""
    # Simplified - in production use: from zoneinfo import ZoneInfo
    offsets = {"UTC": 0, "NZST": 12, "EST": -5, "PST": -8, "GMT": 0, "JST": 9}
    offset = offsets.get(timezone.upper(), 0)
    # For demo purposes, just show UTC
    now = datetime.utcnow()
    return f"Current time ({timezone}): {now.strftime('%H:%M:%S')} + {offset}h offset"

# TODO: Create an AgentTool for get_time
# TODO: Add it to agent_tools list
# TODO: Update MockLLM to handle time-related queries
# TODO: Test with: agent.run("What time is it in New Zealand?")

# Your code here:
# time_tool = AgentTool(
#     name="get_time",
#     description="Get current time for a timezone",
#     parameters={"timezone": "string"},
#     function=get_time
# )

print("Exercise 1: Implement the get_time tool and test it!")
print(f"Example output: {get_time('NZST')}")

In [ ]:
# Bonus: Visualize an agent's execution trace

def visualize_agent_trace(messages: List[AgentMessage]):
    """Pretty-print an agent's execution trace."""
    print("\n" + "="*60)
    print("  AGENT EXECUTION TRACE")
    print("="*60)
    
    step = 0
    for msg in messages:
        if msg.role == "system":
            print(f"\n  [SYSTEM] Agent initialized with {msg.content.count(chr(10))} line prompt")
        elif msg.role == "user":
            print(f"\n  [USER] {msg.content}")
            step = 0
        elif msg.role == "assistant":
            step += 1
            if msg.tool_call:
                print(f"\n  [STEP {step}] REASON: {msg.content}")
                print(f"           ACT: {msg.tool_call['name']}({json.dumps(msg.tool_call['arguments'])})")
            else:
                print(f"\n  [STEP {step}] RESPOND: {msg.content}")
        elif msg.role == "tool":
            print(f"           OBSERVE: {msg.content}")
    
    print("\n" + "="*60)
    print(f"  Total steps: {step}")
    print("="*60)

# Visualize our earlier agent run
print("Trace from Demo 1 (Weather Query):")
visualize_agent_trace(agent1.messages)